# Nordic Research Analytics Pipeline
## Data Engineering I — 1TD169, Uppsala University, Spring 2026

Reads raw NDJSON from HDFS at `/user/nordic/raw`  
Runs all 5 analysis queries against raw data  
Tracks runtime per query — use these numbers for benchmark tables  

**No venv needed** — PySpark reads from HDFS directly via the cluster Spark install.  
Make sure `JAVA_HOME` and `SPARK_HOME` are set before launching Jupyter.

```bash
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export SPARK_HOME=~/spark-3.5.1-bin-hadoop3
export PATH=$SPARK_HOME/bin:$PATH
jupyter notebook --ip=0.0.0.0 --no-browser --port=8888
```

---
## Cell 0 — Cluster Config
**Fill these in before running anything else.**

In [ ]:
# ============================================================
# FILL THESE IN — cluster-specific values
# ============================================================

NAMENODE_PRIVATE_IP  = "192.168.5.39"   # private IP of master node e.g. 192.168.x.x
SPARK_MASTER_IP      = "192.168.5.39"   # same as namenode usually
HDFS_RAW_PATH        = f"hdfs://{NAMENODE_PRIVATE_IP}:9000/user/nordic/raw"
SPARK_MASTER_URL     = f"spark://{SPARK_MASTER_IP}:7077"

# executor config — ssc.medium has 2 vCPU 4GB per node
EXECUTOR_MEMORY      = "2g"
EXECUTOR_CORES       = "1"
DRIVER_MEMORY        = "2g"
SHUFFLE_PARTITIONS   = "8"         # tune this in Experiment 4

print(f"HDFS path : {HDFS_RAW_PATH}")
print(f"Spark URL : {SPARK_MASTER_URL}")

---
## Cell 1 — Imports

In [ ]:
import time

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    explode, col, floor, avg, count, collect_set,
    array_intersect, size, when, sum as spark_sum,
    lit, dense_rank, lag, round as spark_round,
    broadcast, coalesce, percent_rank, ntile,
    stddev, max as spark_max, concat_ws, expr,
    countDistinct
)

NORDIC       = ['SE', 'DK', 'NO', 'FI', 'IS']
RUNTIMES     = {}   # filled as queries run — used for runtime plot at end

print("Imports OK")

---
## Cell 2 — SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("NordicAnalysis_Raw")
    .master(SPARK_MASTER_URL)
    .config("spark.executor.memory",        EXECUTOR_MEMORY)
    .config("spark.executor.cores",         EXECUTOR_CORES)
    .config("spark.driver.memory",          DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
    .config("spark.hadoop.fs.defaultFS",    f"hdfs://{NAMENODE_PRIVATE_IP}:9000")
    .config("spark.hadoop.dfs.replication", "1")   # add this
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

---
## Cell 3 — Load Raw JSON + Schema Check
Validates the data is readable and schema matches expectations  
before any query runs.

In [ ]:
# read raw NDJSON from HDFS
# multiLine=False because OpenAlex snapshot is one JSON object per line
df_raw = spark.read.json(HDFS_RAW_PATH)

print("=== Schema ===")
df_raw.printSchema()

# quick count — confirms data loaded, gives total record number
# this triggers a Spark job so it appears in the Spark UI
t0 = time.time()
total = df_raw.count()
RUNTIMES['count'] = round(time.time() - t0, 2)

print(f"\nTotal records : {total:,}")
print(f"Count job took: {RUNTIMES['count']}s")

# sanity check — show 2 rows, truncated
df_raw.show(2, truncate=80)

---
## Cell 4 — Null Audit
Check how many records have null country codes before any filtering.  
This informs the coalesce guards in every query below.

In [ ]:
# explode once to check null rate on country_code
# this is a diagnostic cell — not part of any query pipeline
null_check = (
    df_raw
    .select(explode('authorships').alias('a'))
    .select(explode('a.institutions').alias('inst'))
    .select(
        col('inst.country_code').alias('cc')
    )
    .agg(
        count('*').alias('total_institution_rows'),
        spark_sum(when(col('cc').isNull(), 1).otherwise(0))
            .alias('null_country_codes'),
        spark_sum(when(col('cc').isin(NORDIC), 1).otherwise(0))
            .alias('nordic_rows')
    )
)

null_check.show()
# expected: ~8% null rate based on API sample validation

---
## Cell 5 — Q1: Institution Output Rankings

**What it answers:** Which Nordic institutions publish the most, ranked per decade?  

Logic:  
- Double explode: works → authorships → institutions  
- Filter to Nordic country codes  
- Group by institution + decade  
- `dense_rank()` within each decade window  
- `lag()` for decade-over-decade growth rate  
- `percent_rank()` for output tier bands

In [ ]:
def query1_institution_output(spark, path):
    df = spark.read.json(path)

    inst_df = (
        df.select('work_id', 'year', explode('authorships').alias('authorship'))
          .select('work_id', 'year', explode('authorship.institutions').alias('inst'))
          .select(
              'work_id',
              # guard nulls before filtering — silent propagation otherwise
              coalesce(col('year'),                  lit(0)).alias('year'),
              coalesce(col('inst.display_name'),     lit('Unknown')).alias('institution'),
              coalesce(col('inst.country_code'),     lit('')).alias('country_code')
          )
          .filter(col('country_code').isin(NORDIC))
          .filter(col('institution') != 'Unknown')
          .withColumn('decade', (floor(col('year') / 10) * 10).cast('int'))
    )

    decade_counts = (
        inst_df.groupBy('institution', 'country_code', 'decade')
               .agg(count('work_id').alias('paper_count'))
    )

    # rank within each decade by volume
    decade_window = Window.partitionBy('decade').orderBy(col('paper_count').desc())
    # growth vs previous decade, per institution
    growth_window = Window.partitionBy('institution').orderBy('decade')

    return (
        decade_counts
        .withColumn('rank_in_decade',    dense_rank().over(decade_window))
        .withColumn('prev_decade_count', lag('paper_count', 1).over(growth_window))
        .withColumn('growth_pct',
            spark_round(
                (col('paper_count') - col('prev_decade_count')) /
                col('prev_decade_count') * 100, 1
            )
        )
        .withColumn('output_percentile', percent_rank().over(decade_window))
        .withColumn('output_tier',
            when(col('output_percentile') >= 0.9, 'Top 10%')
            .when(col('output_percentile') >= 0.5, 'Mid')
            .otherwise('Bottom 50%')
        )
        .orderBy('decade', 'rank_in_decade')
    )


t0 = time.time()
q1 = query1_institution_output(spark, HDFS_RAW_PATH)
q1.show(15, truncate=50)
RUNTIMES['Q1'] = round(time.time() - t0, 2)
print(f"Q1 runtime: {RUNTIMES['Q1']}s")

---
## Cell 6 — Q2: Research Field Trends

**What it answers:** Which fields grew or declined at Nordic institutions by decade?  

Logic:  
- Get Nordic work_ids with decade and country  
- Get topics per work (separate explode on topics array)  
- Join on work_id, register as temp view  
- Spark SQL aggregation with `HAVING COUNT >= 2` to filter single-paper noise  
- `lag()` for growth rate, second `lag()` for momentum (acceleration)

In [ ]:
def query2_field_trends(spark, path):
    df = spark.read.json(path)

    # Step 1: Nordic work_ids with decade + country
    nordic_works = (
        df.select('work_id', 'year', explode('authorships').alias('a'))
          .select('work_id', 'year', explode('a.institutions').alias('i'))
          .select(
              'work_id',
              coalesce(col('year'),           lit(0)).alias('year'),
              coalesce(col('i.country_code'), lit('')).alias('country_code')
          )
          .filter(col('country_code').isin(NORDIC))
          .withColumn('decade', (floor(col('year') / 10) * 10).cast('int'))
          .select('work_id', 'decade', 'country_code')
          .distinct()   # one row per work-country pair
    )

    # Step 2: topics per work — field is a nested struct, guard nulls
    topics = (
        df.select('work_id', explode('topics').alias('topic'))
          .select(
              'work_id',
              coalesce(col('topic.display_name'), lit('Unknown')).alias('field'),
              coalesce(col('topic.field'),        lit('Unknown')).alias('broad_field')
          )
          .filter(col('field') != 'Unknown')
    )

    # Step 3: join and register — Spark SQL paradigm for aggregation
    nordic_works.join(topics, 'work_id') \
                .createOrReplaceTempView('nordic_field_data')

    # Step 4: aggregate — HAVING filters single-paper noise
    # which would create spurious 100%/-100% growth swings in the window
    field_decade = spark.sql("""
        SELECT
            country_code, broad_field, field, decade,
            COUNT(DISTINCT work_id) AS paper_count
        FROM nordic_field_data
        GROUP BY country_code, broad_field, field, decade
        HAVING COUNT(DISTINCT work_id) >= 2
    """)

    # Step 5: growth rate + momentum (2nd derivative of output)
    trend_window = Window.partitionBy('country_code', 'field').orderBy('decade')

    return (
        field_decade
        .withColumn('prev_count',  lag('paper_count', 1).over(trend_window))
        .withColumn('growth_rate',
            spark_round(
                (col('paper_count') - col('prev_count')) /
                col('prev_count') * 100, 1
            )
        )
        .withColumn('prev_growth', lag('growth_rate', 1).over(trend_window))
        # positive momentum = accelerating, negative = decelerating
        .withColumn('momentum',
            spark_round(col('growth_rate') - col('prev_growth'), 1)
        )
        .withColumn('trend_label',
            when(col('momentum') > 10,   'Accelerating')
            .when(col('momentum') < -10,  'Decelerating')
            .when(col('growth_rate') > 0, 'Growing')
            .when(col('growth_rate') < 0, 'Shrinking')
            .otherwise('Stable')
        )
        .orderBy('country_code', 'field', 'decade')
    )


t0 = time.time()
q2 = query2_field_trends(spark, HDFS_RAW_PATH)
q2.show(15, truncate=40)
RUNTIMES['Q2'] = round(time.time() - t0, 2)
print(f"Q2 runtime: {RUNTIMES['Q2']}s")

---
## Cell 7 — Q3: Citation Impact Analysis

**What it answers:** Which Nordic institutions produce the most cited work?  

Logic:  
- Citation counts follow a power-law — avg alone is misleading  
- `stddev()` exposes whether a high average reflects breadth or one outlier  
- Country-level benchmark via `Window.partitionBy('country_code')` — no extra join  
- `ntile(4)` assigns quartiles across all institutions  
- `is_outlier` flags institutions where avg > 2x country mean

In [ ]:
def query3_citation_impact(spark, path):
    df = spark.read.json(path)

    base = (
        df.select('work_id', 'cited_by_count', 'year',
                  explode('authorships').alias('a'))
          .select('work_id',
                  coalesce(col('cited_by_count'), lit(0)).alias('citations'),
                  coalesce(col('year'),           lit(0)).alias('year'),
                  explode('a.institutions').alias('inst'))
          .select('work_id', 'citations', 'year',
                  coalesce(col('inst.display_name'), lit('Unknown')).alias('institution'),
                  coalesce(col('inst.country_code'), lit('')).alias('country_code'))
          .filter(col('country_code').isin(NORDIC))
          .filter(col('institution') != 'Unknown')
    )

    inst_stats = (
        base.groupBy('institution', 'country_code')
            .agg(
                count('work_id').alias('paper_count'),
                spark_round(avg('citations'),    2).alias('avg_citations'),
                spark_round(stddev('citations'), 2).alias('stddev_citations'),
                spark_max('citations').alias('max_citations'),
                # zero-citation count — measures uncited output fraction
                spark_sum(when(col('citations') == 0, 1).otherwise(0))
                    .alias('zero_citation_papers')
            )
            .filter(col('paper_count') >= 5)   # minimum volume — removes noise
    )

    # country benchmark window — computes country avg without extra join/shuffle
    country_window  = Window.partitionBy('country_code')
    quartile_window = Window.orderBy('avg_citations')

    return (
        inst_stats
        .withColumn('country_avg',
            spark_round(avg('avg_citations').over(country_window), 2)
        )
        .withColumn('vs_country_avg',
            spark_round(col('avg_citations') - col('country_avg'), 2)
        )
        # flag institutions with avg > 2x country mean as outliers
        .withColumn('is_outlier',
            col('avg_citations') > col('country_avg') * 2
        )
        .withColumn('citation_quartile', ntile(4).over(quartile_window))
        .withColumn('quartile_label',
            when(col('citation_quartile') == 4, 'Top 25%')
            .when(col('citation_quartile') == 3, 'Upper Mid')
            .when(col('citation_quartile') == 2, 'Lower Mid')
            .otherwise('Bottom 25%')
        )
        .orderBy(col('avg_citations').desc())
    )


t0 = time.time()
q3 = query3_citation_impact(spark, HDFS_RAW_PATH)
q3.show(15, truncate=45)
RUNTIMES['Q3'] = round(time.time() - t0, 2)
print(f"Q3 runtime: {RUNTIMES['Q3']}s")

---
## Cell 8 — Q4: International Collaboration Rate
### PRIMARY BENCHMARK QUERY

**What it answers:** What % of Nordic papers have at least one non-Nordic co-author?  

Logic:  
- `repartition(col('work_id'))` before `collect_set` — puts all rows for a paper  
  on the same partition before aggregation. Critical for shuffle efficiency.  
- `array_intersect` separates Nordic from international countries  
- Exploding `nordic_countries` means SE+DK+UK counts toward both SE and DK rates  
- `collaboration_intensity` = distinct non-Nordic countries per paper  
- `shuffle_partitions` param makes this tunable for Experiment 4

In [ ]:
def query4_international_collab(spark, path, shuffle_partitions=None):
    if shuffle_partitions:
        spark.conf.set('spark.sql.shuffle.partitions', str(shuffle_partitions))

    df = spark.read.json(path)

    # collect all country codes per paper
    # repartition on work_id BEFORE collect_set — colocates rows for same paper
    paper_countries = (
        df.select('work_id', 'year', explode('authorships').alias('a'))
          .select('work_id',
                  coalesce(col('year'), lit(0)).alias('year'),
                  explode('a.institutions').alias('inst'))
          .select('work_id', 'year',
                  coalesce(col('inst.country_code'), lit('')).alias('cc'))
          .filter(col('cc') != '')
          .repartition(col('work_id'))
          .groupBy('work_id', 'year')
          .agg(collect_set('cc').alias('all_countries'))
    )

    nordic_lit       = lit(NORDIC)
    nordic_array_str = ', '.join([f"'{c}'" for c in NORDIC])

    flagged = (
        paper_countries
        .withColumn('decade', (floor(col('year') / 10) * 10).cast('int'))
        .withColumn('nordic_countries',
            array_intersect(col('all_countries'), nordic_lit))
        .withColumn('has_nordic', size(col('nordic_countries')) > 0)
        # international countries = all countries minus Nordic ones
        .withColumn('intl_countries',
            expr(f'filter(all_countries,'
                 f' x -> not array_contains(array({nordic_array_str}), x))'))
        .withColumn('is_international',
            col('has_nordic') & (size(col('intl_countries')) > 0))
        .withColumn('collaboration_intensity',
            size(col('intl_countries')).cast('double'))
        .filter(col('has_nordic'))
    )

    # explode nordic_countries so SE+DK+UK paper counts toward both SE and DK
    per_country = (
        flagged
        .select('work_id', 'decade', 'is_international',
                'collaboration_intensity',
                explode('nordic_countries').alias('nordic_country'))
        .filter(col('nordic_country').isin(NORDIC))
    )

    agg = (
        per_country
        .groupBy('nordic_country', 'decade')
        .agg(
            count('work_id').alias('total_papers'),
            spark_sum(when(col('is_international'), 1).otherwise(0))
                .alias('international_papers'),
            spark_round(
                avg(when(col('is_international'), col('collaboration_intensity'))), 2
            ).alias('avg_intl_countries_when_collab')
        )
        .withColumn('collab_rate',
            spark_round(col('international_papers') / col('total_papers'), 4))
    )

    trend_window = Window.partitionBy('nordic_country').orderBy('decade')

    return (
        agg
        .withColumn('prev_rate',   lag('collab_rate', 1).over(trend_window))
        .withColumn('rate_change',
            spark_round(col('collab_rate') - col('prev_rate'), 4))
        .withColumn('trend',
            when(col('rate_change') > 0.02,   'Increasing')
            .when(col('rate_change') < -0.02,  'Decreasing')
            .when(col('prev_rate').isNull(),   'Baseline')
            .otherwise('Stable')
        )
        .orderBy('nordic_country', 'decade')
    )


t0 = time.time()
q4 = query4_international_collab(spark, HDFS_RAW_PATH)
q4.show(20)
RUNTIMES['Q4'] = round(time.time() - t0, 2)
print(f"Q4 runtime: {RUNTIMES['Q4']}s  <-- use this for benchmark table")

---
## Cell 9 — Q5: Cross-Nordic Collaboration
### HEAVIEST QUERY — Self-join

**What it answers:** How often do institutions across Nordic countries co-author?  

Logic:  
- Naive self-join at author level = O(n²) blowup  
- `.distinct()` before the join deduplicates to one row per paper-country  
- Reduces join input by ~16x (avg 4 authors per paper)  
- `col('a.country') < col('b.country')` canonical ordering prevents double-counting  
- NOTE: broadcast() hint removed for full dataset — 4GB nodes may OOM  
  Add it back only if the deduplicated table is confirmed small

In [ ]:
def query5_cross_nordic(spark, path):
    df = spark.read.json(path)

    # deduplicate to (work_id, decade, country) BEFORE self-join
    # this is the key optimisation — join is on paper-country not author-institution
    nordic_papers = (
        df.select('work_id', 'year', explode('authorships').alias('a'))
          .select('work_id',
                  coalesce(col('year'), lit(0)).alias('year'),
                  explode('a.institutions').alias('inst'))
          .select(
              'work_id',
              (floor(col('year') / 10) * 10).cast('int').alias('decade'),
              coalesce(col('inst.country_code'),  lit('')).alias('country'),
              coalesce(col('inst.display_name'),  lit('Unknown')).alias('institution')
          )
          .filter(col('country').isin(NORDIC))
          .distinct()   # deduplication happens here — before the join
    )

    left  = nordic_papers.alias('a')
    right = nordic_papers.alias('b')

    # no broadcast() hint here — full dataset may OOM on 4GB nodes
    # canonical pair ordering halves output rows and prevents double-counting
    pairs = (
        left.join(right, on=['work_id', 'decade'])
        .filter(col('a.country') < col('b.country'))
        .select(
            'work_id', 'decade',
            col('a.country').alias('country_a'),
            col('b.country').alias('country_b'),
            col('a.institution').alias('inst_a'),
            col('b.institution').alias('inst_b')
        )
    )

    country_pair_decade = (
        pairs
        .groupBy('country_a', 'country_b', 'decade')
        .agg(
            count('work_id').alias('co_authored_papers'),
            countDistinct('inst_a').alias('unique_inst_a'),
            countDistinct('inst_b').alias('unique_inst_b')
        )
    )

    pair_window = Window.partitionBy('country_a', 'country_b')

    return (
        country_pair_decade
        .withColumn('total_pair_papers',
            spark_sum('co_authored_papers').over(pair_window))
        .withColumn('decade_share',
            spark_round(col('co_authored_papers') / col('total_pair_papers'), 3))
        .withColumn('pair_label',
            concat_ws('-', col('country_a'), col('country_b')))
        .orderBy(col('total_pair_papers').desc(), 'decade')
    )


t0 = time.time()
q5 = query5_cross_nordic(spark, HDFS_RAW_PATH)
q5.show(20)
RUNTIMES['Q5'] = round(time.time() - t0, 2)
print(f"Q5 runtime: {RUNTIMES['Q5']}s")

---
## Cell 10 — Runtime Summary
Quick bar chart of all query runtimes.  
Use these numbers in your benchmark tables before Parquet runs.

In [ ]:
import matplotlib.pyplot as plt

print("=== Query Runtimes (Raw JSON on HDFS) ===")
for k, v in RUNTIMES.items():
    print(f"  {k:<8}: {v}s")

# simple runtime bar chart — reference for report
queries = [k for k in RUNTIMES if k != 'count']
times   = [RUNTIMES[k] for k in queries]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(queries, times, color=['#4285f4']*4 + ['#ea4335'],
              edgecolor='white')
ax.set_ylabel('Runtime (seconds)')
ax.set_title('Query Runtimes — Raw JSON, Full Cluster')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{t:.1f}s', ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('query_runtimes_raw.png', dpi=150)
plt.show()
print("Saved to query_runtimes_raw.png")

---
## Cell 11 — Stop Spark

In [ ]:
spark.stop()
print("SparkSession stopped.")